Vectorless RAG for Large Trees — Notes
Problem: Sending the entire tree (e.g., 100,000 nodes) to the LLM is inefficient because it increases token consumption, latency, and cost, and can introduce irrelevant context.

Solution: Retrieve only the relevant nodes before calling the LLM.
Approach 1: BM25 Retriever
1. Indexing: Build a BM25 index from the text of all tree nodes.
2. Query: Receive the user's question.
3. Retrieval: BM25 ranks nodes based on keyword relevance and returns the top-\(K\) nodes.
4. Context building: Include relevant parent or child information when necessary to preserve tree structure.
5. LLM generation: Send only the retrieved nodes and the question to the LLM.
Example:
- Total tree nodes: 100,000
- Retrieved nodes: Top 5
- Nodes sent to LLM: The 5 relevant nodes, plus any necessary context
- Result: Less context, lower token usage, and potentially lower latency.
Important considerations
- rank-bm25: Generally scores the full corpus for each query; suitable for simpler applications and smaller workloads.
- OpenSearch BM25: Uses an inverted index, making it a better option to consider for larger datasets and production workloads.
- BM25 relies on keyword matching, so it may miss relevant nodes when the query uses different terminology.
- Apply relevance thresholds and access-control checks before passing retrieved content to the LLM.
Key takeaway: Retrieve first, generate second. The LLM should receive only the context needed to answer the question, not the entire tree.

Yes. In production, you should design the retrieval system so it can handle growth from 1 lakh to 10 lakh or even millions of nodes without sending the entire tree to the LLM.

The key is to choose an architecture that scales with your data.
Recommended approach
Tree data source

100K → 1M → 10M nodes


Search index

OpenSearch BM25, updated as nodes change


Retriever

Retrieve top-K relevant nodes, e.g. 5–10


Context builder

Add only necessary parent/child information


LLM

Receives a bounded amount of context


What if the data keeps growing?
Concern	Production solution
Number of nodes increases	Use a search engine such as OpenSearch rather than keeping the entire retrieval workload in application memory.
New nodes are added	Update the search index incrementally.
Nodes are updated or deleted	Synchronize updates and deletions with the index.
Search becomes slow	Monitor latency and scale the search cluster, shards, and replicas appropriately.
Too much context is retrieved	Limit top-K and the total context token budget.
Relevant information is missed	Add hybrid retrieval (BM25 + vector search) or query rewriting when needed.
Tree structure matters	Retrieve relevant nodes and selectively include their ancestors or children.

Should we use BM25 or something else?
- rank-bm25 in Python: Good for learning, prototyping, and smaller workloads. It generally scores the corpus for each query, so large datasets and high query volume can become expensive.
- OpenSearch BM25: A stronger starting point for production keyword search at large scale. It uses an inverted index and supports distributed search.
- Hybrid search: Combine BM25 with vector retrieval when users ask semantic questions that may not share keywords with the stored nodes.
One important point: 10 lakh nodes does not automatically mean you need a more complicated architecture. Performance also depends on document size, query volume, hardware, indexing strategy, and latency targets.
My recommendation for your use case
Start with OpenSearch BM25 + bounded top-K retrieval + a context token budget. Measure retrieval latency and relevance using realistic queries and load tests. Add vector retrieval or more advanced routing only if evaluation shows that keyword search is insufficient.

This way, the system can grow without increasing the amount of data sent to the LLM proportionally to the total tree size.

In [ ]:
Yes, OpenSearch can handle millions of nodes and, with the right architecture, even billions of documents. However, “quickly” depends on the size of each node, the query complexity, the number of concurrent users, available hardware, and how the index is configured.

For your vectorless RAG use case, the important distinction is that OpenSearch searches an index of your nodes; it does not send all those nodes to the LLM.
How does it scale?
OpenSearch divides its index into shards, distributes them across multiple servers, and searches relevant shards in parallel. Its inverted index helps find documents matching query terms without scanning every document's full text. 

OpenSearch Documentation
+1



Data size	Practical guidance
100K nodes	A relatively modest search workload; benchmark a small deployment.
1M nodes	OpenSearch is a reasonable option; size resources based on document size and traffic.
10M–100M nodes	Distributed indexing, shard design, and load testing become increasingly important.
1B+ nodes	Possible in principle, but requires serious capacity planning, storage, and operational engineering. Don't assume low latency without benchmarks.

In [ ]:
import json
import re
import numpy as np
from rank_bm25 import BM25Okapi
from openai import OpenAI

client = OpenAI()  # Set OPENAI_API_KEY in your environment


# Example tree data; replace this with your 100,000 nodes.
nodes = [
    {
        "id": "1",
        "parent_id": None,
        "text": "Company policies"
    },
    {
        "id": "2",
        "parent_id": "1",
        "text": "Leave policy"
    },
    {
        "id": "3",
        "parent_id": "2",
        "text": "Employees receive 20 days of annual leave."
    },
    {
        "id": "4",
        "parent_id": "1",
        "text": "Expense policy"
    },
    {
        "id": "5",
        "parent_id": "4",
        "text": "Employees must submit expense claims within 30 days."
    }
]


def tokenize(text):
    return re.findall(r"\b[a-z0-9]+\b", text.lower())


# Build lookup maps and the searchable corpus.
nodes_by_id = {node["id"]: node for node in nodes}

tokenized_corpus = [
    tokenize(node["text"])
    for node in nodes
]

# Build the index ONCE, not for every user query.
bm25 = BM25Okapi(tokenized_corpus)


def retrieve_nodes(query, top_k=5):
    query_tokens = tokenize(query)

    if not query_tokens:
        return []

    # BM25 calculates relevance scores for the corpus.
    scores = bm25.get_scores(query_tokens)

    # Get the indexes of the top-k scoring nodes.
    top_indexes = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indexes:
        node = nodes[index]
        score = float(scores[index])

        results.append({
            "id": node["id"],
            "parent_id": node["parent_id"],
            "text": node["text"],
            "score": score
        })

    return results


def build_context(retrieved_nodes):
    """Add parent information to help preserve tree context."""
    context_parts = []
    added_ids = set()

    for node in retrieved_nodes:
        parent_id = node["parent_id"]

        if parent_id and parent_id not in added_ids:
            parent = nodes_by_id.get(parent_id)

            if parent:
                context_parts.append(
                    f"Parent node [{parent['id']}]: {parent['text']}"
                )
                added_ids.add(parent_id)

        context_parts.append(
            f"Node [{node['id']}]: {node['text']}"
        )

    return "\n".join(context_parts)


def ask_llm(question, context):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using the supplied context. "
                    "If the context does not contain the answer, "
                    "say that you cannot determine the answer "
                    "from the available context."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Question: {question}"
                )
            }
        ]
    )

    return response.choices[0].message.content


def answer_question(question):
    # Step 1: Retrieve relevant nodes.
    retrieved = retrieve_nodes(question, top_k=5)

    if not retrieved:
        return "No relevant nodes were found."

    # Step 2: Build context from retrieved nodes.
    context = build_context(retrieved)

    # Step 3: Send only retrieved context to the LLM.
    answer = ask_llm(question, context)

    return answer


if __name__ == "__main__":
    question = "How many days of annual leave do employees receive?"

    retrieved = retrieve_nodes(question, top_k=3)

    print("Retrieved nodes:")
    for node in retrieved:
        print(node)

    print("\nLLM answer:")
    print(answer_question(question))